    FILE:  pyThreeD.ipynb
    DATE:  12 DEC 2023
    AUTH:  G. E. Deschaines
    DESC:  Three dimensional (3D) drawing of missile/target engagement trajectory
           objects defined as collection of polygons.
    REFS:

    [1] This Python script was translated from threeD.c available at:
        https://github.com/gedeschaines/threeD/blob/master/src/threeD.c

    Disclaimer:  See DISCLAIMER file.

In [ ]:
import sys
import os

RUN_WITH_COLAB = False   # Set to True when run with Google Colab

if RUN_WITH_COLAB:
    from google.colab import widgets
    from google.colab import output
    output.enable_custom_widget_manager()

    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
        # The following statements assumes pyThreeD.ipynb and draw3D.py
        # are both in the specified Google Drive folder.
        Google_Drive_Folder = '/content/drive/MyDrive/Colab Notebooks/propNav'
        if Google_Drive_Folder not in sys.path:
            sys.path.insert(0, Google_Drive_Folder)
        os.chdir(Google_Drive_Folder)
    except:
        pass

import time
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib

Backend = 'Qt5Agg'  # to display rendering in desktop window.
#Backend = 'nbAgg'   # to display rendering animation inline.

if RUN_WITH_COLAB:
    Backend = 'nbAgg'
    
matplotlib.use(Backend.lower())

# Note: If 'nbAgg' is specified for the Backend and a JavaScript "IPython is not defined"
#       error message appears below, then refer to the following for possible resolution.
#       https://stackoverflow.com/questions/51922480/javascript-error-ipython-is-not-defined-in-jupyterlab

if Backend == 'nbAgg':
    # Enable the ipympl backend
    try:
        %matplotlib widget
    except:
        %pip install -q ipympl
        %matplotlib widget

import matplotlib.pyplot as plt

In [ ]:
global theFigure  # Instantiated matplotlib pyplot figure
global theDraw3D  # Instantiated Draw3D object
global started    # Draw3D.MainLoop method started flag

from draw3D import Draw3D

started = False

In [ ]:
def onClick(event):
    """
    Mouse button pressed handler
    """
    global theFigure
    global theDraw3D
    global started

    #b = event.button
    #x = event.xdata
    #y = event.ydata
    if not started:
       started = True
       theDraw3D.MainLoop()
       if theDraw3D.doneflag:
           theFigure.clear()
           plt.close(theFigure)
           sys.exit()
       started = False

In [ ]:
# Set image save flag and rate (FPS).

img_Save = False  # Save rendered images as PNG files
img_FPS = 50;     # (semi-colon is hold over from threeD.c)

# Set TXYZ case specifics.

CaseId = "1243"
File   = "./out/TXYZ.OUT." + CaseId
title  = "pyThreeD - " + File

In [ ]:
# Instantiate the matplotlib pyplot figure for display of
# engagement rendering animation.

# Note: A Figure window will be opened on the desktop
#       for rendering animation if the Backend = 'nbAgg'
#       statement in Cell [1] above is commented.

# For 100 dots-per-inch (dpi):
#figsize=(6.0,6.0)     # 465x462 viewport
figsize=(8.0,6.0)      # 620x462 viewport
#figsize=(8.0,8.0)     # 620x616 viewport
#figsize=(7.74,7.78)   # 600x599 viewport
#figsize=(10.0,8.0)    # 775x616 viewport
#figsize=(10.32,7.80)  # 800x601 viewport

if Backend == 'nbAgg':
    with plt.ioff():
        theFigure = plt.figure(figsize=figsize, dpi=100.0)
else:
    theFigure = plt.figure(figsize=figsize, dpi=100.0)

try:
    theFigure.set_tight_layout(True)
except:
    theFigure.set_layout_engine('tight')

# Change the toolbar position
#theFigure.canvas.toolbar_position = 'top'
# Always hide the toolbar
theFigure.canvas.toolbar_visible = False
# Disable the resizing feature
theFigure.canvas.resizable = False

if Backend == 'nbAgg':
    # If true then scrolling while the mouse is over the canvas
    # will not move the entire notebook.
    theFigure.canvas.capture_scroll = False


In [ ]:
# Specify plotting layout and parameters.

ax = theFigure.add_subplot(111, autoscale_on=False, animated=False)
ax.set_title(title)
ax.set_xticks([])
ax.set_yticks([])
ax.set_aspect('equal')
bbox = ax.bbox.get_points()
w = round(bbox[1,0] - bbox[0,0])
h = round(bbox[1,1] - bbox[0,1])
ax.set_xlim(0.0, w)
ax.set_ylim(h, 0.0)
if Backend != 'nbAgg':
    ax.set_facecolor('#d8dcd6')  # light gray
theFigure.canvas.draw()

In [ ]:
# Instantiate the Draw3D object to perform missile/target
# engagement rendering.

msltyp = int(CaseId)//1000
theDraw3D = Draw3D(txyzFile=File, mslType=msltyp,
                   w=w, h=h, fig=theFigure, ax=ax,
                   imgSave=img_Save, imgFPS=img_FPS)

# Load object shape polygon data.

theDraw3D.ReadPolyData()

In [ ]:
# Assign mouse button press handler.

cidbtn = theFigure.canvas.mpl_connect('button_press_event', onClick)

In [ ]:
if Backend != 'nbAgg':
    # Assign GUI key press event handler.
    cidkey = theFigure.canvas.mpl_connect('key_press_event', theDraw3D.onPress)

    # Present mouse click and key press instructions:
    print("With cursor in the displayed Figure, press a mouse button")
    print("to initiate threeD animation. If animation is exited or has")
    print("completed, close figure to terminate this program or press")
    print("a mouse button to restart animation. During threeD program")
    print("animation the following key presses are recognized:\n")
    print("Press T key to toggle field-of-view towards target.")
    print("Press M key to toggle field-of-view towards missile.")
    print("Press H key to toggle field-of-view along missile heading.")
    print("Press Z key to reset zoom to one.")
    print("Press Up Arrow key to increase zoom.")
    print("Press Down Arrow key to decrease zoom.")
    print("Press Left Arrow key to slow animation down by 10 msec increments.")
    print("Press Right Arrow key to speed animation up by 10 msec increments.")
    print("Press Space key to toggle pause/unpause.")
    print("Press X key to exit animation (press a mouse button to restart).")
    print("Press Q key to close Figure and exit program.")

    # Display figure window for rendering and wait for
    # user mouse click and key presses...
    plt.show(block=True)

    # Execution terminated, delete the Draw3D object.
    del theDraw3D

else:
    print("Click in the plotting region of Figure 1 presented")
    print("immediately below the following cell to start or")
    print("replay rendering animation.\n")
    print("Inline rendering animation cannot be controlled with")
    print("key presses as they are intercepted and processed by")
    print("this Jupyter notebook and matplotlib widget backend.")


In [ ]:
# Note: Figure 1 for displaying rendering animation will
#       be presented immediately below this Cell if the
#       Backend = 'nbAgg' statement in Cell [1] above
#       is uncommented.

if Backend == 'nbAgg':
    display(theFigure.canvas)